# Сравнение детекторов аномалий траектории

In [1]:
TEST_SPLIT_1 = 0.4
TEST_SPLIT_2 = 0.2

## Чтение и предобработка данных

Исходный файл csv задаёт последовательность записей с полями времени, координат и статуса. После чтения строится очищенный ряд наблюдений и две пары выборок «обучение / тест» с разными долями обучения.

После загрузки CSV имеем индексы $j=0,\ldots,M-1$. Сырые поля: $t_j$ (time), $\varphi_j$ (lat), $\lambda_j$ (lon), $y_j$ (status). Допустимые значения метки: $y_j\in\{\mathrm{move},\mathrm{anomaly},\ldots\}$; в эксперименте остаются только $\mathrm{move}$ и $\mathrm{anomaly}$. Для метрик классификации положительный класс — аномалия ($y_i=\mathrm{anomaly}$); в столбце label она кодируется нулём, штатный ход — единицей.

**1. Отбор столбцов.** Оставляем только
$$(t_i,\varphi_i,\lambda_i,y_i)_{i=0}^{N-1},$$
где $t_i$ — время, $\varphi_i$ — широта, $\lambda_i$ — долгота, $y_i$ — метка класса.

**2. Валидные координаты.** Исключаются пропуски, нечисловые координаты, выход за допустимые пределы WGS-84 и точки «нулевого» GPS:
$$\mathcal{J}_1=\{i:\ \varphi_i,\lambda_i\ \text{конечны},\ -90\le\varphi_i\le 90,\ -180\le\lambda_i\le 180,\ (\varphi_i,\lambda_i)\neq(0,0)\}.$$

**3. Бинарная задача.** Остаются только штатный ход и аномалия:
$$\mathcal{J}_2=\{i\in\mathcal{J}_1:\ y_i\in\{\mathrm{move},\mathrm{anomaly}\}\}.$$

**4. Упорядочивание.** На $\mathcal{J}_2$ задаётся сортировка по возрастанию $t_i$. Индекс $i=0$ — первая валидная точка ряда.

**5. Локальная ENU (м).** Опорная точка — первая валидная точка: $(\varphi_0,\lambda_0)=(\varphi_{0},\lambda_{0})$ в нумерации после шагов 1–4. Координаты WGS-84 переводятся в плоскость «восток–север» (WGS-84 $\to$ ECEF $\to$ ENU):
$$(x_i,n_i)\equiv(E_i,N_i)=\mathrm{ENU}(\varphi_i,\lambda_i\mid\varphi_0,\lambda_0).$$
Поля $\varphi_i,\lambda_i$ сохраняются; столбцы $x_i$, $n_i$ добавляются в таблицу (метка класса по-прежнему $y_i$).

**6. Обрезка по крайним $\mathrm{move}$.** Пусть
$$\mathcal{I}_{\mathrm{move}}=\{i:\ y_i=\mathrm{move}\},\quad i_{\min}=\min\mathcal{I}_{\mathrm{move}},\quad i_{\max}=\max\mathcal{I}_{\mathrm{move}}.$$
Рабочий ряд — срез $i_{\min}\le i\le i_{\max}$ (первая и последняя точки ряда — $\mathrm{move}$); координаты $(x_i,n_i)$ не пересчитываются.

**7. Разбиение train / test.** Для доли обучения $\rho\in(0,1)$ (в блокноте $\rho=\texttt{TEST\_SPLIT\_1}$ или $\texttt{TEST\_SPLIT\_2}$) при длине ряда $N$ задаётся пороговый индекс
$$k_0=\lfloor\rho N\rfloor.$$
Граница разбиения — первый индекс штатного хода не левее $k_0$:
$$k=\min\{i\ge k_0:\ y_i=\mathrm{move}\}.$$
Если $y_{k_0}=\mathrm{move}$, то $k=k_0$. Обучение: индексы $[0,k)$, тест: $[k,N)$. Так граница не попадает внутрь чисто аномального фрагмента без соседнего $\mathrm{move}$.

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.special import logsumexp
from scipy.stats import chi2
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.svm import OneClassSVM
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from app.evaluation import GeospatialMetrics, TrajectoryMetrics, TrajectorySeries
from app.geo.local_enu import LocalEnuConverter
from app.viz import plot_track_maps

TORCH_CUDA_INDEX = "https://download.pytorch.org/whl/cu124"


def find_root() -> Path:
    """Ищет корень репозитория по наличию data/2.csv и пакета app.

    Returns:
        Path: абсолютный путь к корню проекта.

    Raises:
        FileNotFoundError: если корень не найден среди cwd и родительских каталогов.
    """
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "2.csv").exists() and (candidate / "app").exists():
            return candidate
    raise FileNotFoundError("Не найден корень проекта с data/2.csv")


path_root = find_root()
path_data = path_root / "data" / "2.csv"
if str(path_root) not in sys.path:
    sys.path.insert(0, str(path_root))


def ensure_torch_cuda() -> None:
    """Устанавливает сборку PyTorch с CUDA, если GPU NVIDIA есть, а wheel собран как +cpu.

    Returns:
        None: при уже доступной CUDA или отсутствии nvidia-smi.

    Raises:
        RuntimeError: после успешной установки — нужен перезапуск ядра блокнота.
        subprocess.CalledProcessError: если pip не смог установить пакеты.
    """
    import torch

    if torch.cuda.is_available():
        return
    if shutil.which("nvidia-smi") is None:
        return
    cpu_wheel = "+cpu" in torch.__version__ or not torch.backends.cuda.is_built()
    if not cpu_wheel:
        return
    print("Обнаружен GPU NVIDIA, но PyTorch без CUDA. Устанавливаю torch с cu124...")
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--upgrade",
            "torch",
            "torchvision",
            "torchaudio",
            "--index-url",
            TORCH_CUDA_INDEX,
        ]
    )
    raise RuntimeError(
        "PyTorch с CUDA установлен. Перезапустите ядро блокнота и выполните ячейку снова."
    )


ensure_torch_cuda()

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    DEVICE = torch.device("cpu")
torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))

print(f"Корень: {path_root}")
print(f"Файл: {path_data}")
print(f"PyTorch {torch.__version__}, устройство {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(DEVICE)}")
elif shutil.which("nvidia-smi") is not None:
    print(
        "NVIDIA GPU есть, но torch.cuda.is_available()=False. "
        f"Проверьте сборку (ожидается не +cpu) или переустановите: pip install torch --index-url {TORCH_CUDA_INDEX}"
    )

Обнаружен GPU NVIDIA, но PyTorch без CUDA. Устанавливаю torch с cu124...


In [ ]:
ALLOWED_STATUS = ("move", "anomaly")


def nfmt(value: int) -> str:
    """Форматирует целое число с разделителем тысяч (пробел).

    Args:
        value: целое значение для вывода.

    Returns:
        str: строка вида «1 234 567».
    """
    return f"{int(value):,}".replace(",", " ")


def preprocess_raw_frame(raw: pd.DataFrame) -> pd.DataFrame:
    """Предобработка сырого CSV: фильтры, ENU, обрезка по крайним move (см. markdown выше).

    Args:
        raw: таблица из data/2.csv со столбцами time, lat, lon, status.

    Returns:
        pd.DataFrame: столбцы time, lat, lon, y, x, n, label (anomaly → 0, move → 1).

    Raises:
        RuntimeError: если после фильтрации не осталось точек move.
    """
    frame = raw.loc[:, ["time", "lat", "lon", "status"]].rename(columns={"status": "y"}).copy()
    frame["lat"] = pd.to_numeric(frame["lat"], errors="coerce")
    frame["lon"] = pd.to_numeric(frame["lon"], errors="coerce")
    valid = frame["lat"].notna() & frame["lon"].notna()
    valid &= frame["lat"].between(-90.0, 90.0) & frame["lon"].between(-180.0, 180.0)
    valid &= ~((frame["lat"] == 0.0) & (frame["lon"] == 0.0))
    frame = frame.loc[valid].reset_index(drop=True)
    labels = frame["y"].astype(str)
    frame = frame.loc[labels.isin(ALLOWED_STATUS)].reset_index(drop=True)
    order = np.argsort(pd.to_datetime(frame["time"]).to_numpy(), kind="mergesort")
    frame = frame.iloc[order].reset_index(drop=True)
    enu = LocalEnuConverter.from_first_point(frame["lat"].iloc[0], frame["lon"].iloc[0])
    east, north = enu.to_local_xy(frame["lat"].to_numpy(), frame["lon"].to_numpy())
    frame["x"] = east
    frame["n"] = north
    is_move = frame["y"].astype(str).to_numpy() == "move"
    if not np.any(is_move):
        raise RuntimeError("После фильтрации нет точек move.")
    move_idx = np.flatnonzero(is_move)
    frame = frame.iloc[move_idx[0] : move_idx[-1] + 1].reset_index(drop=True)
    frame["label"] = np.where(frame["y"].astype(str) == "anomaly", 0.0, 1.0)
    return frame


def split_train_test_move(frame: pd.DataFrame, train_fraction: float) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Делит ряд на обучение [0, k) и тест [k, N); граница k — первый move с индексом i >= floor(rho*N).

    Args:
        frame: предобработанный ряд с столбцом y (move / anomaly).
        train_fraction: доля обучения rho из (0, 1), как TEST_SPLIT_1 или TEST_SPLIT_2.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: (train, test).

    Raises:
        ValueError: если train_fraction не в (0, 1).
        RuntimeError: если ряд слишком короткий, нет подходящего move или выборка пуста.
    """
    if not 0.0 < train_fraction < 1.0:
        raise ValueError("train_fraction должен быть в (0, 1).")
    n = len(frame)
    if n < 2:
        raise RuntimeError("Слишком короткий ряд для разбиения.")
    k0 = int(np.floor(train_fraction * n))
    k0 = min(max(k0, 1), n - 1)
    labels = frame["y"].astype(str).to_numpy()
    move_from_k0 = np.flatnonzero((np.arange(n) >= k0) & (labels == "move"))
    if move_from_k0.size == 0:
        raise RuntimeError(f"Нет move для границы при rho={train_fraction}.")
    k = int(move_from_k0[0])
    train = frame.iloc[:k].reset_index(drop=True)
    test = frame.iloc[k:].reset_index(drop=True)
    if len(train) == 0 or len(test) == 0:
        raise RuntimeError(f"Пустая выборка при rho={train_fraction}, k={k}, N={n}.")
    return train, test


def describe_split(name: str, train: pd.DataFrame, test: pd.DataFrame) -> None:
    """Печатает размер и доли классов anomaly и move для обучения и теста.

    Args:
        name: заголовок сценария (например, df1 и доля rho).
        train: обучающая выборка.
        test: тестовая выборка.
    """
    for part_name, part in (("обучение", train), ("тест", test)):
        labels = part["y"].astype(str)
        n = len(part)
        p_anom = float((labels == "anomaly").mean())
        p_move = float((labels == "move").mean())
        print(
            f"{name}: {part_name:<9} {nfmt(n):>10} точек   "
            f"anomaly {p_anom:6.1%}   move {p_move:6.1%}"
        )


raw_full = pd.read_csv(path_data)
frame = preprocess_raw_frame(raw_full)
train_1, test_1 = split_train_test_move(frame, TEST_SPLIT_1)
train_2, test_2 = split_train_test_move(frame, TEST_SPLIT_2)

df1 = {"train": train_1, "test": test_1}
df2 = {"train": train_2, "test": test_2}

print(f"Исходный CSV: {nfmt(len(raw_full))} строк, после предобработки: {nfmt(len(frame))}.")
describe_split("df1 (rho={:.0%})".format(TEST_SPLIT_1), train_1, test_1)
describe_split("df2 (rho={:.0%})".format(TEST_SPLIT_2), train_2, test_2)

## Используемые метрики

### Коэффициент корреляции Мэттьюса (MCC)

**Коэффициент корреляции Мэттьюса** (Matthews correlation coefficient, MCC) — коэффициент корреляции Пирсона между двумя бинарными векторами: эталонной разметкой и предсказанием (каждая компонента кодируется как 0 или 1). Эквивалентно он выражается через четыре числа таблицы сопряжённости $TP$, $FP$, $FN$, $TN$ и принимает значения от $-1$ до $1$.

**Используется в задачах** бинарной классификации, в том числе при сильном дисбалансе классов: оценка предсказаний структуры белка, медицинская диагностика, обнаружение аномалий, сравнение порогов и моделей, когда accuracy и доля «большого» класса малоинформативны.

**Источник:** Chicco D., Jurman G. *The advantages of the Matthews correlation coefficient (MCC) over F1 score and accuracy in binary classification evaluation* // BMC Genomics. 2020. Vol. 21, Art. 6. Open access: [10.1186/s12864-019-6413-7](https://link.springer.com/article/10.1186/s12864-019-6413-7). (В статье отмечено, что MCC изначально предложен B. W. Matthews в 1975 г.; в статье приведены математические свойства и сравнение с accuracy и $F_1$.)

**Основное преимущество:** метрика одновременно учитывает все четыре ячейки таблицы сопряжённости и симметрично отражает цену ложных срабатываний и пропусков; при дисбалансе классов тривиальные предсказания «всегда отрицательный» или «всегда положительный» класс обычно дают MCC около нуля, тогда как accuracy может оставаться высокой.

**Для положительного класса** (в этом блокноте положительный класс — **аномалия**, $y_i=0$, бинарная метка $a_i=\mathbf{1}[y_i=0]$; предсказание $\hat a_i=\mathbf{1}[\hat y_i=0]$):

$$
\mathrm{MCC}=\frac{TP\cdot TN - FP\cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}},
$$
где:\
$TP=\sum_i \mathbf{1}[a_i=1\land \hat a_i=1]$,\
$FP=\sum_i \mathbf{1}[a_i=0\land \hat a_i=1]$,\
$FN=\sum_i \mathbf{1}[a_i=1\land \hat a_i=0]$,\
$TN=\sum_i \mathbf{1}[a_i=0\land \hat a_i=0]$.

**Интерпретация:** $\mathrm{MCC}\in[-1,1]$; $1$ — полное совпадение масок; $0$ — согласие на уровне случайного предсказания при тех же маргинальных долях классов; отрицательные значения — систематическое расхождение хуже случайного. Если знаменатель равен нулю (например, в выборке или в предсказании только один класс), MCC в строгом смысле не определён; в коде этого блокнота в таком случае принимается $0$.

### Distance loss

Геопространственная метрика: насколько расходятся длины эталонной и предсказанной траектории (в метрах). Длина ломаной — сумма геодезических звеньев $d_{\mathrm{geo}}$; distance loss — модуль разности:

$$
L(T)=\sum_i d_{\mathrm{geo}}(p_i,p_{i+1}), \qquad DL=\bigl|L_{\mathrm{pred}}-L_{\mathrm{ref}}\bigr|.
$$

$L_{\mathrm{ref}}$ строится по $G$ (штатные точки и линейная интерполяция в аномалиях между соседними штатными). $L_{\mathrm{pred}}$ — по $P$: в ломаную входят только точки, принятые за штатный ход ($\hat y_i\neq 0$); пропуски не интерполируются, соседние принятые точки соединяются напрямую. На тесте $DL$ также считается через на заполненных траекториях; при выборе порога по MCC в полосе близкого значения MCC минимизируется операционный distance loss (длина по принятым move-точкам).